# 02 — Triple-Barrier Labeling & Meta-Labeling

**AFML Chapter 3** — López de Prado (2018), pp. 43–67

Standard ML labeling ("did price go up?") has two problems:
1. **Fixed horizon** ignores path — a trade that hit -20% before recovering is labeled the same as one that went straight up
2. **No bet sizing** — all trades are treated equally

The **triple-barrier method** fixes (1) by defining three barriers:
- Upper (profit-taking): take profit at +X vol
- Lower (stop-loss): cut losses at -Y vol  
- Vertical (time expiry): if neither barrier is hit in N bars, label = 0

**Meta-labeling** fixes (2) by decomposing the decision into:
1. A *primary model* chooses the side (long/short)
2. A *secondary model* decides whether to take the bet (and how much)

---

In [ ]:
from _setup import *
from afml.labeling import (
    daily_volatility, make_events, triple_barrier_labels,
    meta_labels, fixed_horizon_labels,
)
import duckdb

## 1. Load data & compute daily volatility

In [ ]:
panel = load_daily_bars()
panel = filter_universe(panel)

btc = panel[panel["symbol"] == "BTC-USD"].sort_values("ts").drop_duplicates("ts", keep="last")
close = btc.set_index("ts")["close"]
print(f"BTC daily: {len(close)} bars, {close.index.min().date()} to {close.index.max().date()}")

vol = daily_volatility(close, span=20)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(close.index, close.values, color=NAVY, lw=0.8)
axes[0].set_ylabel("BTC Close ($)")
axes[0].set_title("BTC-USD Price & Daily Volatility", fontweight="bold")
axes[0].set_yscale("log")

axes[1].plot(vol.index, vol.values * 100, color=RED, lw=0.8)
axes[1].axhline(vol.median() * 100, color=GRAY, ls="--", lw=0.8,
                label=f"Median: {vol.median()*100:.1f}%")
axes[1].set_ylabel("Daily Vol (%)")
axes[1].legend(fontsize=9)
fig.tight_layout()
plt.show()

## 2. Triple-barrier labeling

We set profit-taking and stop-loss at **2x daily vol** with a
**10-day** vertical barrier (max holding period).

This means barriers adapt to the current volatility regime:
- Low-vol periods: tighter barriers (e.g. ±2%)
- High-vol periods: wider barriers (e.g. ±10%)

In [ ]:
events = make_events(close, vol, holding_periods=10)
tb = triple_barrier_labels(close, events, pt_sl=(2.0, 2.0))

print(f"Events: {len(events):,}  |  Labeled: {len(tb):,}")
print(f"\nLabel distribution:")
label_counts = tb["label"].value_counts().sort_index()
label_names = {-1: "Stop-loss", 0: "Time expiry", 1: "Profit-take"}
for lbl, cnt in label_counts.items():
    print(f"  {label_names[lbl]:15s}  {cnt:>5,}  ({cnt/len(tb):.1%})")

In [ ]:
# Visualise labels on price chart (2023-2024 window)
fig, ax = plt.subplots(figsize=(14, 6))

window = (tb.index >= "2023-01-01") & (tb.index < "2025-01-01")
tb_w = tb[window]
close_w = close.loc["2023-01-01":"2025-01-01"]

ax.plot(close_w.index, close_w.values, color=GRAY, lw=0.8, alpha=0.7, label="BTC")

colors_lbl = {1: GREEN, -1: RED, 0: GOLD}
names_lbl = {1: "Profit-take", -1: "Stop-loss", 0: "Expiry"}
for lbl in [1, -1, 0]:
    mask = tb_w["label"] == lbl
    entries = tb_w[mask]
    ax.scatter(entries.index, close.reindex(entries.index),
              c=colors_lbl[lbl], s=12, alpha=0.6, label=names_lbl[lbl], zorder=3)

ax.set_title("Triple-Barrier Labels on BTC (2023–2024)", fontweight="bold")
ax.set_ylabel("Price ($)")
ax.legend(fontsize=9, loc="upper left")
fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "02_triple_barrier_labels.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Triple barrier vs fixed horizon

Compare the label distributions: triple-barrier labels are more
balanced and capture path-dependent outcomes that fixed-horizon misses.

In [ ]:
fh_labels = fixed_horizon_labels(close, horizon=10, method="sign")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Fixed horizon
ax = axes[0]
fh_vc = fh_labels.value_counts().sort_index()
ax.bar(fh_vc.index, fh_vc.values, color=[RED, GOLD, GREEN], width=0.5)
ax.set_xticks([-1, 0, 1])
ax.set_xticklabels(["Down", "Flat", "Up"])
ax.set_title("Fixed Horizon (10d)", fontweight="bold")
ax.set_ylabel("Count")
for i, (x, v) in enumerate(zip(fh_vc.index, fh_vc.values)):
    ax.text(x, v + 10, f"{v/len(fh_labels):.0%}", ha="center", fontsize=10)

# Triple barrier
ax = axes[1]
tb_vc = tb["label"].value_counts().sort_index()
ax.bar(tb_vc.index, tb_vc.values, color=[RED, GOLD, GREEN], width=0.5)
ax.set_xticks([-1, 0, 1])
ax.set_xticklabels(["Stop-loss", "Expiry", "Profit-take"])
ax.set_title("Triple Barrier (2x vol, 10d)", fontweight="bold")
for i, (x, v) in enumerate(zip(tb_vc.index, tb_vc.values)):
    ax.text(x, v + 10, f"{v/len(tb):.0%}", ha="center", fontsize=10)

fig.suptitle("Labeling Methods Compared", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "02_label_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Sensitivity to barrier width

How do labels change as we vary the profit-take / stop-loss multiplier?
Wider barriers → more time expiries, fewer stop-outs.

In [ ]:
multipliers = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0]
sensitivity = []

for m in multipliers:
    tb_m = triple_barrier_labels(close, events, pt_sl=(m, m))
    vc = tb_m["label"].value_counts()
    sensitivity.append({
        "multiplier": m,
        "pct_profit_take": vc.get(1, 0) / len(tb_m),
        "pct_stop_loss": vc.get(-1, 0) / len(tb_m),
        "pct_expiry": vc.get(0, 0) / len(tb_m),
        "avg_ret_winners": tb_m[tb_m["label"] == 1]["ret"].mean(),
        "avg_ret_losers": tb_m[tb_m["label"] == -1]["ret"].mean(),
    })

sens_df = pd.DataFrame(sensitivity).set_index("multiplier")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(sens_df.index, sens_df["pct_profit_take"] * 100, "o-", color=GREEN, label="Profit-take")
ax.plot(sens_df.index, sens_df["pct_stop_loss"] * 100, "o-", color=RED, label="Stop-loss")
ax.plot(sens_df.index, sens_df["pct_expiry"] * 100, "o-", color=GOLD, label="Expiry")
ax.set_xlabel("Barrier multiplier (× daily vol)")
ax.set_ylabel("% of labels")
ax.set_title("Label Distribution vs Barrier Width", fontweight="bold")
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(sens_df.index, sens_df["avg_ret_winners"] * 100, "o-", color=GREEN, label="Avg winner return")
ax.plot(sens_df.index, sens_df["avg_ret_losers"] * 100, "o-", color=RED, label="Avg loser return")
ax.axhline(0, color=GRAY, ls="-", lw=0.5)
ax.set_xlabel("Barrier multiplier (× daily vol)")
ax.set_ylabel("Average return (%)")
ax.set_title("Return Magnitude vs Barrier Width", fontweight="bold")
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "02_barrier_sensitivity.png"), dpi=150, bbox_inches="tight")
plt.show()

sens_df.style.format("{:.1%}")

## 5. Meta-labeling with MA crossover

We use a simple **MA 20/50 crossover** as the primary model to decide
the side (long/short), then ask: *"given the primary model's side,
was the trade profitable?"*

This binary meta-label (1 = profitable, 0 = not) becomes the target
for a secondary ML model that learns to filter and size bets.

In [ ]:
# Primary model: MA crossover
ma_fast = close.rolling(20).mean()
ma_slow = close.rolling(50).mean()
primary_side = pd.Series(np.where(ma_fast > ma_slow, 1.0, -1.0), index=close.index)

# Triple barrier with the primary side
events_meta = make_events(close, vol, holding_periods=10, side=primary_side)
tb_meta = triple_barrier_labels(close, events_meta, pt_sl=(2.0, 2.0))

# Meta-labels
ml = meta_labels(primary_side, tb_meta)

print(f"Meta-labeling results (MA 20/50 primary):")
print(f"  Total observations:  {len(ml):,}")
print(f"  Hit rate (meta=1):   {ml['meta_label'].mean():.1%}")
print(f"  Avg return (meta=1): {ml[ml['meta_label']==1]['ret'].mean():+.4f}")
print(f"  Avg return (meta=0): {ml[ml['meta_label']==0]['ret'].mean():+.4f}")

In [ ]:
# Distribution of returns by meta-label
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(ml[ml["meta_label"] == 1]["ret"], bins=50, alpha=0.7, color=GREEN,
        label=f"Meta=1 (n={int((ml['meta_label']==1).sum())})", density=True)
ax.hist(ml[ml["meta_label"] == 0]["ret"], bins=50, alpha=0.5, color=RED,
        label=f"Meta=0 (n={int((ml['meta_label']==0).sum())})", density=True)
ax.set_xlabel("Signed Return")
ax.set_ylabel("Density")
ax.set_title("Return Distribution by Meta-Label", fontweight="bold")
ax.legend(fontsize=9)

# Rolling hit rate
ax = axes[1]
ml_indexed = ml.copy()
ml_indexed.index = pd.to_datetime(ml_indexed.index)
rolling_hr = ml_indexed["meta_label"].rolling(120).mean()
ax.plot(rolling_hr.index, rolling_hr.values * 100, color=NAVY, lw=1.0)
ax.axhline(50, color=GRAY, ls="--", lw=0.8)
ax.axhline(ml["meta_label"].mean() * 100, color=TEAL, ls="-.", lw=0.8,
           label=f"Overall: {ml['meta_label'].mean():.1%}")
ax.set_ylabel("Hit Rate (%)")
ax.set_title("Rolling 120-bar Hit Rate (MA 20/50 Primary)", fontweight="bold")
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "02_meta_labeling.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Multi-asset triple-barrier analysis

Apply triple-barrier labeling across the full universe to see how
label distributions vary by asset.

In [ ]:
# Use top symbols by data coverage
sym_counts = panel[panel["in_universe"]].groupby("symbol").size()
top_syms = sym_counts.nlargest(15).index.tolist()

multi_results = []
for sym in top_syms:
    s = panel[panel["symbol"] == sym].sort_values("ts").drop_duplicates("ts", keep="last")
    c = s.set_index("ts")["close"]
    if len(c) < 100:
        continue

    v = daily_volatility(c, span=20)
    ev = make_events(c, v, holding_periods=10)
    tb_s = triple_barrier_labels(c, ev, pt_sl=(2.0, 2.0))

    if len(tb_s) == 0:
        continue

    vc = tb_s["label"].value_counts()
    multi_results.append({
        "symbol": sym,
        "n_labels": len(tb_s),
        "pct_pt": vc.get(1, 0) / len(tb_s),
        "pct_sl": vc.get(-1, 0) / len(tb_s),
        "pct_exp": vc.get(0, 0) / len(tb_s),
        "median_vol": v.median(),
        "avg_ret_pt": tb_s[tb_s["label"] == 1]["ret"].mean() if (tb_s["label"] == 1).any() else np.nan,
        "avg_ret_sl": tb_s[tb_s["label"] == -1]["ret"].mean() if (tb_s["label"] == -1).any() else np.nan,
    })

multi_df = pd.DataFrame(multi_results).set_index("symbol")
display(multi_df.style.format({
    "pct_pt": "{:.1%}", "pct_sl": "{:.1%}", "pct_exp": "{:.1%}",
    "median_vol": "{:.3f}", "avg_ret_pt": "{:+.3f}", "avg_ret_sl": "{:+.3f}",
}))

In [ ]:
# Scatter: median vol vs hit rate (profit-take %)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(multi_df["median_vol"] * 100, multi_df["pct_pt"] * 100,
           s=60, color=NAVY, alpha=0.7, zorder=3)
for sym in multi_df.index:
    ax.annotate(sym.replace("-USD", ""), 
               (multi_df.at[sym, "median_vol"] * 100, multi_df.at[sym, "pct_pt"] * 100),
               fontsize=8, ha="left", va="bottom", xytext=(3, 3), textcoords="offset points")

ax.set_xlabel("Median Daily Volatility (%)")
ax.set_ylabel("Profit-Take Rate (%)")
ax.set_title("Profit-Take Rate vs Volatility (2x vol barriers, 10d expiry)", fontweight="bold")
fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "02_vol_vs_hitrate.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Summary

**Key findings from applying AFML Chapter 3 to crypto:**

1. **Triple-barrier labels** produce more informative labels than fixed-horizon.
   Path-dependent outcomes (hit stop-loss then recovered) are correctly handled.

2. **Dynamic barriers** (scaled by daily vol) adapt naturally to crypto's
   regime-switching volatility — tight barriers in calm periods, wide in crises.

3. **Meta-labeling** the MA 20/50 crossover shows ~55% hit rate with
   meaningful return asymmetry. This is the input for a secondary ML model
   that can learn *when* the primary signal works and *how much* to bet.

4. **Cross-asset patterns**: higher-vol coins tend to have lower profit-take
   rates (harder to reach 2x vol barrier before stop-loss), confirming that
   barrier width must scale with the asset's characteristics.

The labeling module provides the foundation for:
- **Ch 4**: Sample weights (uniqueness of overlapping labels)
- **Ch 6**: Training ensemble models on triple-barrier labels
- **Ch 10**: Bet sizing from meta-label probabilities

---

*Next: [03_fracdiff.ipynb](03_fracdiff.ipynb) — Fractionally Differentiated Features (AFML Ch 5)*